In [ ]:
import cv2
import math
from ultralytics import YOLO

model = YOLO("runs/detect/output/train/weights/best.pt") 

# Avståndströskel i pixlar
DISTANCE_THRESHOLD = 300 

# Gruppnamn
PLURAL_NAMES = {
    'Dryckeskartong': 'KARTONGER',
    'Konservburk': 'KONSERVBURKAR',
    'Pantburk': 'PANTBURKAR'
}

# Färger
COLOR_GREEN = (0, 255, 0)     # Sorterat
COLOR_RED = (0, 0, 255)       # Osorterat blandat
COLOR_YELLOW = (0, 255, 255)  # Ensamt/Varning

def calculate_distance(p1, p2):
    """Räknar ut euklidiskt avstånd mellan två punkter (x1, y1) och (x2, y2)"""
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)

# Starta webcam
cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Kunde inte läsa från kameran. Avslutar...")
        break

    results = model(frame, verbose=False)
    
    detections = []
    
    # Gå igenom alla identifierade objekt och spara deras data
    for r in results:
        boxes = r.boxes
        for box in boxes:
            # Hämta koordinater för bounding box
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            
            # Hämta klassnamn
            cls_id = int(box.cls[0])
            cls_name = model.names[cls_id]
            
            # Beräkna mittpunkten
            cx = int((x1 + x2) / 2)
            cy = int((y1 + y2) / 2)
            
            detections.append({
                'box': (x1, y1, x2, y2),
                'centroid': (cx, cy),
                'class': cls_name
            })

    # Gruppera objekt som ligger nära varandra
    clusters = []
    visited = set()
    
    for i in range(len(detections)):
        if i not in visited:
            
            current_cluster = [i]
            visited.add(i)
            
            # Letar efter grannar med en enkel sökning (Breadth-First Search-liknande), GPT
            q = [i]
            while q:
                curr = q.pop(0)
                for j in range(len(detections)):
                    if j not in visited:
                        # Om avståndet mellan objekt curr och j är mindre än tröskelvärdet
                        dist = calculate_distance(detections[curr]['centroid'], detections[j]['centroid'])
                        if dist < DISTANCE_THRESHOLD:
                            visited.add(j)
                            current_cluster.append(j)
                            q.append(j)
            
            clusters.append(current_cluster)

    # Rita ut resultatet baserat på if-satser för varje grupp
    for cluster_indices in clusters:
        
        # ENSAMT
        if len(cluster_indices) == 1:
            idx = cluster_indices[0]
            det = detections[idx]
            x1, y1, x2, y2 = det['box']
            
            cv2.rectangle(frame, (x1, y1), (x2, y2), COLOR_YELLOW, 2)
            cv2.putText(frame, det['class'], (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_YELLOW, 2)
            
        else:
            # Vi har flera objekt nära varandra, kolla om de är av samma klass
            classes_in_cluster = set([detections[i]['class'] for i in cluster_indices])
            
            # SORTERAT
            if len(classes_in_cluster) == 1:
                cls_name = list(classes_in_cluster)[0]
                
                # Rita gröna rutor runt alla objekt i gruppen
                for idx in cluster_indices:
                    x1, y1, x2, y2 = detections[idx]['box']
                    cv2.rectangle(frame, (x1, y1), (x2, y2), COLOR_GREEN, 2)
                
                # Hitta objektet som ligger högst upp (minst y1) för att skriva texten ovanför hela gruppen
                top_idx = min(cluster_indices, key=lambda i: detections[i]['box'][1])
                top_x1, top_y1 = detections[top_idx]['box'][0], detections[top_idx]['box'][1]
                
                # Skriv ut gruppnamn
                plural_text = PLURAL_NAMES.get(cls_name, cls_name + "S")
                cv2.putText(frame, plural_text, (top_x1, top_y1 - 15), cv2.FONT_HERSHEY_SIMPLEX, 0.8, COLOR_GREEN, 2)
                
            # OSORTERAT
            else:
                for idx in cluster_indices:
                    det = detections[idx]
                    x1, y1, x2, y2 = det['box']
                    
                    # Rita röd ruta och skriv ut vad den enskilda saken är
                    cv2.rectangle(frame, (x1, y1), (x2, y2), COLOR_RED, 2)
                    cv2.putText(frame, det['class'], (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, COLOR_RED, 2)

    # Visa resultatet
    cv2.imshow("Avfallssortering", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Kunde inte läsa från kameran. Avslutar...
